# 🔬  AI Data Concierge - Reproducible Analysis

<a href="https://colab.research.google.com/" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

---

## 📋 Query
> **Which Pittsburgh neighborhoods have the most police incidents?**

## 📊 Metadata
| Property | Value |
|----------|-------|
| **Generated** | 2026-04-03 14:04:22 |
| **Data Source** | WPRDC |
| **Sources Used** | Western PA Regional Data Center (WPRDC) |
| **Notebook Version** | 1.0 (Colab Compatible) |

---

## 📖 How to Use This Notebook

This notebook reproduces the exact analysis performed by the ** AI Data Concierge**.
Follow the steps below to verify, modify, or extend the analysis.

### ✅ Quick Start
1. **Run All Cells**: Click `Runtime` → `Run all` (or press `Ctrl+F9`)
2. **Wait for Setup**: The first cells install dependencies and configure the environment

### 🔧 What You Can Do
| Action | Description |
|--------|-------------|
| **Verify** | Run all cells to confirm the original results |
| **Modify** | Change parameters (dates, locations, filters) and re-run |
| **Extend** | Add your own analysis cells below the results |
| **Export** | Download results as CSV, or save notebook to Drive |

### 📚 Notebook Structure
1. **Setup** - Install dependencies (runs once in Colab)
2. **Configuration** - Import libraries and set up API connections
3. **Data Retrieval** - Fetch data from the data source
4. **Analysis** - Process and analyze the data
5. **Results** - View the final answer and confidence scores
6. **Citations** - Reference sources for your research

---


In [ ]:
# ============================================================
# STEP 1: Environment Setup
# ============================================================
# This cell installs all required packages for Google Colab.
# If running locally, you can skip this cell if packages are installed.

# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("🔧 Running in Google Colab - Installing dependencies...")
    !pip install -q pandas numpy matplotlib seaborn requests
    print("✅ Dependencies installed successfully!")
else:
    print("💻 Running locally - assuming dependencies are installed")
    print("   If not, run: pip install pandas numpy matplotlib seaborn requests")


In [ ]:
# ============================================================
# STEP 2: Import Libraries
# ============================================================
# These are the core libraries used throughout this notebook.
# Each import serves a specific purpose in the data pipeline.

import requests      # For making HTTP requests to data APIs
import pandas as pd  # For data manipulation and analysis
import numpy as np   # For numerical computations
from datetime import datetime  # For timestamp handling
import json          # For JSON parsing

# Visualization libraries (with graceful fallback)
try:
    import matplotlib.pyplot as plt  # For creating plots
    import seaborn as sns            # For statistical visualizations
    VISUALIZATION_AVAILABLE = True
    plt.style.use('seaborn-v0_8-whitegrid')
    print("📊 Visualization libraries loaded successfully")
except ImportError:
    VISUALIZATION_AVAILABLE = False
    print("⚠️ Visualization libraries not available")
    print("   Install with: pip install matplotlib seaborn")

# ============================================================
# STEP 3: Configuration
# ============================================================
# Data source configuration - modify these if needed

CKAN_URL = "https://data.wprdc.org"

# Display configuration info
print(f"\n📅 Notebook generated: {datetime.now().isoformat()}")
print(f"🔗 CKAN URL: {CKAN_URL}")
print(f"📌 Timestamp: 2026-04-03T14:04:22.589668")


In [ ]:
# ============================================================
# STEP 4: Helper Functions
# ============================================================
# These utility functions handle data fetching from the CKAN API.
# You can reuse these functions for your own data exploration.

def fetch_ckan_data(resource_id: str, limit: int = 10000, filters: dict = None) -> pd.DataFrame:
    """
    Fetch data from CKAN DataStore API.

    This function handles pagination automatically and returns all records
    up to the specified limit.

    Parameters:
    -----------
    resource_id : str
        The unique identifier for the CKAN resource (dataset)
    limit : int, default=10000
        Maximum number of records to fetch
    filters : dict, optional
        Field filters to apply (e.g., {"state": "CA"})

    Returns:
    --------
    pd.DataFrame
        A DataFrame containing the fetched records

    Example:
    --------
    >>> df = fetch_ckan_data("abc123", limit=1000, filters={"year": 2023})
    >>> print(f"Loaded {len(df)} records")
    """
    print(f"📥 Fetching data from resource: {resource_id}")

    all_records = []
    offset = 0
    batch_size = min(32000, limit)

    while offset < limit:
        params = {
            "resource_id": resource_id,
            "limit": min(batch_size, limit - offset),
            "offset": offset,
        }

        if filters:
            params["filters"] = filters

        response = requests.post(
            f"{CKAN_URL}/api/3/action/datastore_search",
            json=params,
            headers={"Content-Type": "application/json"},
        )

        if response.status_code != 200:
            print(f"❌ Error: {response.status_code} - {response.text}")
            break

        result = response.json()
        if not result.get("success"):
            print(f"❌ CKAN error: {result.get('error')}")
            break

        records = result.get("result", {}).get("records", [])
        if not records:
            break

        all_records.extend(records)
        offset += len(records)

        total = result.get("result", {}).get("total", 0)
        print(f"   Progress: {len(all_records):,} / {total:,} records")

        if offset >= total:
            break

    df = pd.DataFrame(all_records)

    # Remove CKAN internal fields that aren't useful for analysis
    internal_cols = ["_id", "_full_text"]
    df = df.drop(columns=[c for c in internal_cols if c in df.columns], errors="ignore")

    print(f"✅ Loaded {len(df):,} records with {len(df.columns)} columns")
    return df


def search_ckan_packages(query: str, rows: int = 10) -> list:
    """
    Search CKAN for datasets (packages) matching a query.

    Parameters:
    -----------
    query : str
        The search query (e.g., "employment statistics")
    rows : int, default=10
        Maximum number of results to return

    Returns:
    --------
    list
        A list of matching package dictionaries

    Example:
    --------
    >>> packages = search_ckan_packages("housing prices", rows=5)
    >>> for pkg in packages:
    ...     print(pkg['title'])
    """
    print(f"🔍 Searching for: '{query}'")

    response = requests.post(
        f"{CKAN_URL}/api/3/action/package_search",
        json={"q": query, "rows": rows},
        headers={"Content-Type": "application/json"},
    )

    if response.status_code != 200:
        print(f"❌ Search failed: {response.status_code}")
        return []

    result = response.json()
    if not result.get("success"):
        print(f"❌ Search error: {result.get('error')}")
        return []

    packages = result.get("result", {}).get("results", [])
    print(f"✅ Found {len(packages)} matching datasets")
    return packages


print("✅ Helper functions loaded successfully!")
print("   - fetch_ckan_data(resource_id, limit, filters)")
print("   - search_ckan_packages(query, rows)")


# Data Analysis Pipeline

The following steps show how the data was searched, loaded, and analyzed. Each step includes executable code you can modify and re-run.

---

## Step 1: Search for Datasets

**Search query:** `police incidents`
**Organization:** `city-of-pittsburgh`

**Result preview:**
```
Found 7 datasets matching 'police incidents'

1. **Police Incident Blotter (Archived)**
   ID: `uniform-crime-reporting-data`
   **This data ceased updating with the transition to a new records management system on 11/14/2023. Access to the updated data set has been added as of April 11, 2025 here: <a href="
   - Blotter Data (UCR Coded) (CSV) [DataStore] ID: `044f2016-1dfd-4ab0-bc1e-065da05fca2e`
   - Archived Police Incident Blotter Data Dictionary (XLSX) [DataStore] ID: `a0e233b3-8cfc-441a-a37e-d396579d20ea`
   - Historical Blotter Data (CSV) [DataStore] ID: `391942e2-25ef-43e4-8263-f8519fa
```


In [ ]:
# Step 1: Search for Datasets

# Search for datasets
import requests, json

params = {"q": 'police incidents', "rows": 10, "fq": "organization:city-of-pittsburgh"}
resp = requests.get("https://data.wprdc.org/api/3/action/package_search", params=params)
results = resp.json()["result"]["results"]

print(f"Found {resp.json()['result']['count']} datasets")
for i, ds in enumerate(results, 1):
    print(f"\n{i}. {ds['title']}")
    print(f"   ID: {ds['name']}")
    for r in ds.get("resources", []):
        print(f"   - {r['name']} ({r['format']}) ID: {r['id']}")


## Step 2: SQL Analysis Query

**SQL:**
```sql
SELECT "INCIDENTNEIGHBORHOOD", COUNT(*) as incident_count FROM "044f2016-1dfd-4ab0-bc1e-065da05fca2e" GROUP BY "INCIDENTNEIGHBORHOOD" ORDER BY incident_count DESC LIMIT 20
```

**Result preview:**
```
SQL: SELECT "INCIDENTNEIGHBORHOOD", COUNT(*) as incident_count FROM "044f2016-1dfd-4ab0-bc1e-065da05fca2e" GROUP BY "INCIDENTNEIGHBORHOOD" ORDER BY incident_count DESC LIMIT 20
Rows: 20
Columns: INCIDENTNEIGHBORHOOD, incident_count

     INCIDENTNEIGHBORHOOD  incident_count
Central Business District           19943
         South Side Flats           19098
                  Carrick           11927
                Shadyside            8773
               Bloomfield            8704
             East Liberty            8451
      Squirrel Hill South            7809
                Brookline      
```


In [ ]:
# Step 2: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "INCIDENTNEIGHBORHOOD", COUNT(*) as incident_count FROM "044f2016-1dfd-4ab0-bc1e-065da05fca2e" GROUP BY "INCIDENTNEIGHBORHOOD" ORDER BY incident_count DESC LIMIT 20'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 3: SQL Analysis Query

**SQL:**
```sql
SELECT "NEIGHBORHOOD", COUNT(*) as incident_count FROM "bd41992a-987a-4cca-8798-fbe1cd946b07" GROUP BY "NEIGHBORHOOD" ORDER BY incident_count DESC LIMIT 20
```

**Result preview:**
```
SQL error (HTTP 409): {"help": "https://data.wprdc.org/api/3/action/help_show?name=datastore_search_sql", "error": {"query": ["(psycopg2.errors.UndefinedColumn) column \"NEIGHBORHOOD\" does not exist\nLINE 1: ...LAIN (VERBOSE, FORMAT JSON) SELECT * FROM (SELECT \"NEIGHBORH...\n                                                             ^\n\n[SQL: SELECT * FROM (SELECT \"NEIGHBORHOOD\", COUNT(*) as incident_count FROM \"bd41992a-987a-4cca-8798-fbe1cd946b07\" GROUP BY \"NEIGHBORHOOD\" ORDER BY incident_count DESC LIMI
```


In [ ]:
# Step 3: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "NEIGHBORHOOD", COUNT(*) as incident_count FROM "bd41992a-987a-4cca-8798-fbe1cd946b07" GROUP BY "NEIGHBORHOOD" ORDER BY incident_count DESC LIMIT 20'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 4: Load Data from Resource

**Resource ID:** `bd41992a-987a-4cca-8798-fbe1cd946b07`
**Limit:** 5

**Result preview:**
```
Resource: bd41992a-987a-4cca-8798-fbe1cd946b07
Total records: 89,038
Loaded: 5
Fields (19): Report_Number, ReportedDate, ReportedTime, Hour, DayofWeek, ReportedMonth, NIBRS_Coded_Offense, NIBRS_Offense_Code, NIBRS_Offense_Category, NIBRS_Offense_Type, NIBRS_Crime_Against, NIBRS_Offense_Grouping, Violation, XCOORD, YCOORD, Zone, Tract, Neighborhood, Block_Address

Sample (5 rows):

Report_Number ReportedDate ReportedTime  Hour DayofWeek ReportedMonth                      NIBRS_Coded_Offense NIBRS_Offense_Code NIBRS_Offense_Category   NIBRS_Offense_Type NIBRS_Crime_Against NIBRS_Offense_Grouping
```


In [ ]:
# Step 4: Load Data from Resource

# Load resource data
import pandas as pd

params = {"resource_id": 'bd41992a-987a-4cca-8798-fbe1cd946b07', "limit": 5}
resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search", json=params)
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Total available: {result['total']:,}")
print(f"\nColumns: {\", \".join(df.columns.tolist())}")
df.head(10)


## Step 5: SQL Analysis Query

**SQL:**
```sql
SELECT COUNT(*) as total FROM "044f2016-1dfd-4ab0-bc1e-065da05fca2e"
```

**Result preview:**
```
SQL: SELECT COUNT(*) as total FROM "044f2016-1dfd-4ab0-bc1e-065da05fca2e"
Rows: 1
Columns: total

 total
340996
```


In [ ]:
# Step 5: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT COUNT(*) as total FROM "044f2016-1dfd-4ab0-bc1e-065da05fca2e"'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


## Step 6: SQL Analysis Query

**SQL:**
```sql
SELECT "Neighborhood", COUNT(*) as incident_count FROM "bd41992a-987a-4cca-8798-fbe1cd946b07" GROUP BY "Neighborhood" ORDER BY incident_count DESC LIMIT 20
```

**Result preview:**
```
SQL: SELECT "Neighborhood", COUNT(*) as incident_count FROM "bd41992a-987a-4cca-8798-fbe1cd946b07" GROUP BY "Neighborhood" ORDER BY incident_count DESC LIMIT 20
Rows: 20
Columns: Neighborhood, incident_count

             Neighborhood  incident_count
Central Business District            9132
                      NaN            8051
         South Side Flats            5681
                  Carrick            2814
             East Liberty            2437
                Shadyside            2390
           East Allegheny            2295
                Knoxville            1773
             
```


In [ ]:
# Step 6: SQL Analysis Query

# Run SQL analysis query
sql = 'SELECT "Neighborhood", COUNT(*) as incident_count FROM "bd41992a-987a-4cca-8798-fbe1cd946b07" GROUP BY "Neighborhood" ORDER BY incident_count DESC LIMIT 20'

resp = requests.post("https://data.wprdc.org/api/3/action/datastore_search_sql", json={"sql": sql})
result = resp.json()["result"]

df = pd.DataFrame(result["records"])
df = df.drop(columns=["_id", "_full_text"], errors="ignore")

print(f"Query returned {len(df)} rows")
df


# ============================================================
# 📊 RESULTS
# ============================================================

## 💡 Answer

> **Here's a comprehensive breakdown of Pittsburgh neighborhoods with the most police incidents, based on two datasets from the **Western PA Regional Data Center (WPRDC)**:

---

## 🚨 Pittsburgh Neighborhoods with the Most Police Incidents

### 📁 Historical Data (Archived Blotter — through Nov 2023)
*341,000 total incident records*

| Rank | Neighborhood | Incidents |
|------|-------------|-----------|
| 1 | **Central Business District** | 19,943 |
| 2 | **South Side Flats** | 19,098 |
| 3 | **Carrick** | 11,927 |
| 4 | **Shadyside** | 8,773 |
| 5 | **Bloomfield** | 8,704 |
| 6 | **East Liberty** | 8,451 |
| 7 | **Squirrel Hill South** | 7,809 |
| 8 | **Brookline** | 7,792 |
| 9 | **Mount Washington** | 7,723 |
| 10 | **Homewood South** | 7,297 |

---

### 📁 Recent Data (2024–2026 Monthly Criminal Activity)
*89,000 total incident records*

| Rank | Neighborhood | Incidents |
|------|-------------|-----------|
| 1 | **Central Business District** | 9,132 |
| 2 | **South Side Flats** | 5,681 |
| 3 | **Carrick** | 2,814 |
| 4 | **East Liberty** | 2,437 |
| 5 | **Shadyside** | 2,390 |
| 6 | **East Allegheny** | 2,295 |
| 7 | **Knoxville** | 1,773 |
| 8 | **Bloomfield** | 1,764 |
| 9 | **Mount Washington** | 1,743 |
| 10 | **Squirrel Hill South** | 1,609 |

---

### 🔑 Key Takeaways

- **Central Business District** and **South Side Flats** are consistently the top two neighborhoods for police incidents across both time periods — likely reflecting their high foot traffic, nightlife, and density.
- **Carrick, Shadyside, Bloomfield, East Liberty, and Mount Washington** also appear in the top 10 in both datasets, suggesting persistently elevated incident rates.
- **Homewood South** ranks high historically but drops slightly in recent years, while **East Allegheny** has become more prominent recently.

### ⚠️ Caveats
- The historical dataset spans many years, so raw counts are much higher.
- ~6,900–8,000 records have **no neighborhood listed** (NaN), which may slightly undercount some areas.
- Incident counts include all crime types and are *not* normalized by population — denser/busier areas naturally see more incidents.

**Source:** Pittsburgh Police Incident Blotter (Archived) & Monthly Criminal Activity Dataset — [data.wprdc.org](https://data.wprdc.org)**

---

## 🎯 Confidence Assessment

The Data Concierge evaluates the reliability of its answer using multiple factors:


> ⚠️ Confidence score not available for this query.


# ============================================================
# 📚 CITATIONS & REFERENCES
# ============================================================

## 📖 Data Sources

## Data Sources and Citations

**[1]** Western PA Regional Data Center (WPRDC)
- Dataset: Open Data Portal
- URL: [https://data.wprdc.org](https://data.wprdc.org)
- Accessed: 2026-04-03

---

## 🔄 Reproducibility Guide

This notebook was automatically generated by the ** AI Data Concierge**.
Follow these steps to reproduce or extend the analysis:

### Prerequisites

```bash
pip install pandas numpy requests matplotlib seaborn
```

### Running the Notebook

| Step | Action | Notes |
|------|--------|-------|
| 1 | **Open in Colab** | Click the "Open in Colab" badge at the top |
| 2 | **Run All Cells** | `Runtime` → `Run all` or `Ctrl+F9` |
| 3 | **Wait for completion** | Dependencies install automatically in Colab |
| 4 | **Review results** | Scroll down to see the analysis results |

### ⚠️ Important Notes

- **Data freshness**: Results may differ if data sources have been updated since generation
- **API limits**: Some data sources have rate limits; wait if you encounter errors
- **Modifications**: Feel free to modify parameters and re-run cells to explore further

### 📅 Generation Info

- **Generated**: 2026-04-03 14:04:22
- **Query**: Which Pittsburgh neighborhoods have the most police incidents?
- **Data Source**: WPRDC

---

### 💬 Need Help?

- **Documentation**: [Data Concierge Guide](https://github.com/dathere/data-concierge)
- **Issues**: Report problems or request features on GitHub
- **Extend**: Add your own analysis cells below this one!

---

*Generated by  AI Data Concierge v0.1.0*
